# Notebook 01 — Détection d'anomalies (Non supervisé)

**Objectif** : Détecter automatiquement les événements anormaux en course F1  
(crashes, sorties de piste, safety car, voitures lentes) via **Isolation Forest** et **DBSCAN**.

**Données** : FastF1 — tours, télémétrie, statut de piste  
**Modèles** : Isolation Forest + DBSCAN (comparaison)


In [ ]:
import sys
sys.path.insert(0, '..')  # Accès aux modules src/

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from src.data_loader import load_session, get_laps_features, get_weather
from src.unsupervised import (
    detect_isolation_forest,
    detect_dbscan,
    compare_models,
    plot_anomalies_timeline,
    plot_feature_distribution,
    print_anomaly_report,
)

plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Chargement de la session

In [ ]:
# Paramètres — modifiez selon vos besoins
YEAR = 2025
GP_NAME = 'Monaco'
SESSION_TYPE = 'R'  # R = Race

session = load_session(YEAR, GP_NAME, SESSION_TYPE)
if session is None:
    print(f'  [INFO] {YEAR} {GP_NAME} indisponible — fallback 2024')
    YEAR = 2024
    session = load_session(YEAR, GP_NAME, SESSION_TYPE)
print(f'Session chargée : {session.event["EventName"]} {YEAR}' if session else 'Erreur de chargement')

In [ ]:
laps_df = get_laps_features(session)
print(f'Nombre de tours : {len(laps_df)}')
print(f'Pilotes : {laps_df["Driver"].unique()}')
laps_df.head()

## 2. Exploration des données

In [ ]:
print('Statistiques descriptives :')
laps_df[['LapTime_s', 'TyreLife', 'SpeedST', 'SpeedFL', 'LapTimeDelta']].describe()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
cols = ['LapTime_s', 'LapTimeDelta', 'TyreLife', 'SpeedST', 'SpeedFL', 'Position']
axes = axes.flatten()

for i, col in enumerate(cols):
    if col in laps_df.columns:
        axes[i].hist(laps_df[col].dropna(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
        axes[i].set_title(col)
        axes[i].grid(True, alpha=0.3)

plt.suptitle(f'Distribution des features — {YEAR} GP {GP_NAME}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Détection avec Isolation Forest

In [ ]:
df_if = detect_isolation_forest(laps_df, contamination=0.05)

print(f"Anomalies détectées (IF) : {df_if['IF_Anomaly'].sum()} / {len(df_if)} tours")
df_if[df_if['IF_Anomaly']][['Driver', 'LapNumber', 'LapTime_s', 'IF_Score']].sort_values('IF_Score').head(15)

## 4. Détection avec DBSCAN

In [ ]:
df_dbscan = detect_dbscan(laps_df, eps=1.5, min_samples=5)

print(f"Anomalies détectées (DBSCAN) : {df_dbscan['DBSCAN_Anomaly'].sum()} / {len(df_dbscan)} tours")
print(f"Clusters formés : {df_dbscan['DBSCAN_Label'].nunique() - 1}")
df_dbscan[df_dbscan['DBSCAN_Anomaly']][['Driver', 'LapNumber', 'LapTime_s']].head(15)

## 5. Comparaison des deux modèles

In [ ]:
df_annotated = compare_models(laps_df)
print_anomaly_report(df_annotated)

## 6. Visualisation sur la timeline de course

In [ ]:
fig = plot_anomalies_timeline(df_annotated, title=f'Anomalies — {YEAR} GP {GP_NAME}')
plt.show()

## 7. Distribution des features — Normal vs Anomalie

In [ ]:
fig = plot_feature_distribution(df_annotated)
plt.show()

## 8. Analyse par pilote

In [ ]:
driver_anomalies = (
    df_annotated.groupby('Driver')[['IF_Anomaly', 'DBSCAN_Anomaly', 'Both_Anomaly']]
    .sum()
    .sort_values('IF_Anomaly', ascending=False)
)
driver_anomalies

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(driver_anomalies))
width = 0.35
ax.bar([i - width/2 for i in x], driver_anomalies['IF_Anomaly'], width, label='Isolation Forest', color='#e74c3c', alpha=0.8)
ax.bar([i + width/2 for i in x], driver_anomalies['DBSCAN_Anomaly'], width, label='DBSCAN', color='#3498db', alpha=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(driver_anomalies.index, rotation=45, ha='right')
ax.set_ylabel('Nombre de tours anormaux')
ax.set_title(f'Anomalies par pilote — {YEAR} GP {GP_NAME}', fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Correspondance avec les événements réels (Safety Car, VSC)

> FastF1 fournit `TrackStatus` : 1=Normal, 2=Yellow, 4=Safety Car, 6=Virtual SC, 7=Red Flag

In [ ]:
if 'TrackStatus' in df_annotated.columns:
    status_map = {'1': 'Normal', '2': 'Yellow', '4': 'Safety Car', '6': 'Virtual SC', '7': 'Red Flag'}
    df_annotated['TrackStatusLabel'] = df_annotated['TrackStatus'].astype(str).map(status_map).fillna('Autre')
    
    cross = pd.crosstab(df_annotated['TrackStatusLabel'], df_annotated['IF_Anomaly'],
                        margins=True, margins_name='Total')
    cross.columns = ['Normal', 'Anomalie', 'Total']
    display(cross)
    
    print('\nTaux de détection par statut piste :')
    for status in df_annotated['TrackStatusLabel'].unique():
        subset = df_annotated[df_annotated['TrackStatusLabel'] == status]
        rate = 100 * subset['IF_Anomaly'].mean()
        print(f'  {status:15s}: {rate:.1f}% détecté comme anomalie')
else:
    print('TrackStatus non disponible pour cette session.')